In [1]:
import json
import ast
from pathlib import Path

In [2]:
DATASET_FOLDER = Path(
    "/Users/megan/Downloads/PMC_Patients"
)

OUTPUT_FOLDER = DATASET_FOLDER / "tfidf_output"

TASK2_FILE = OUTPUT_FOLDER / "task2_similar_patients.json"

OUTPUT_FILE = OUTPUT_FOLDER / "task3_evaluation_results.json"

SUMMARY_FILE = OUTPUT_FOLDER / "task3_evaluation_summary.json"

In [3]:
with open(TASK2_FILE, "r", encoding="utf-8") as file:
    task2_results = json.load(file)

print("Task 2 results loaded successfully.")
print("Number of patients:", len(task2_results))

Task 2 results loaded successfully.
Number of patients: 8579


In [6]:
ground_truth = {}
original_patient_ids = {}

json_files = sorted(DATASET_FOLDER.rglob("*.json"))

for file_path in json_files:

    
    if OUTPUT_FOLDER in file_path.parents:
        continue

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)

    except (json.JSONDecodeError, OSError) as error:
        print(f"Could not read {file_path}: {error}")
        continue

    
    records = data if isinstance(data, list) else [data]

    for record in records:

        if not isinstance(record, dict):
            continue

        patient_uid = record.get("patient_uid")
        patient_id = record.get("patient_id")

        if not patient_uid:
            continue

        patient_uid = str(patient_uid)
        patient_id = str(patient_id or "")

        # similar_patients
        similar = record.get("similar_patients", {})

        # dictionaries
        if isinstance(similar, str):
            similar = similar.strip()

            if similar:
                similar = ast.literal_eval(similar)
            else:
                similar = {}

        if similar is None:
            similar = {}

        if not isinstance(similar, dict):
            raise ValueError(
                f"Invalid similar_patients structure "
                f"for patient {patient_uid}."
            )

        # scores 1 and 2 are counted equally
        relevant_patients = set()

        for similar_uid, relevance in similar.items():

            if relevance in (1, 2):
                relevant_patients.add(str(similar_uid))

        if patient_uid in ground_truth:
            raise ValueError(
                f"Duplicate patient_uid found: {patient_uid}"
            )

        ground_truth[patient_uid] = relevant_patients
        original_patient_ids[patient_uid] = patient_id


print("Ground truth extracted successfully.")
print("Patients with ground-truth records:", len(ground_truth))

Ground truth extracted successfully.
Patients with ground-truth records: 8579


In [7]:
task2_patient_uids = set(task2_results.keys())
ground_truth_patient_uids = set(ground_truth.keys())

if task2_patient_uids != ground_truth_patient_uids:

    missing_from_task2 = (
        ground_truth_patient_uids - task2_patient_uids
    )

    missing_from_ground_truth = (
        task2_patient_uids - ground_truth_patient_uids
    )

    raise ValueError(
        "Patient UID mismatch detected.\n"
        f"Missing from Task 2: {len(missing_from_task2)}\n"
        f"Missing from ground truth: "
        f"{len(missing_from_ground_truth)}"
    )

print("Patient matching successful.")
print("Total patients:", len(ground_truth))

Patient matching successful.
Total patients: 8579


In [8]:
def calculate_metrics(retrieved_patients, relevant_patients):

    # top five retrieved patients
    top_5 = retrieved_patients[:5]

    # count relevant patients
    relevant_retrieved = len(
        set(top_5) & relevant_patients
    )

    # precision at 5
    precision_5 = relevant_retrieved / 5

    # recall at 5
    if len(relevant_patients) == 0:
        recall_5 = None

    else:
        recall_5 = (
            relevant_retrieved / len(relevant_patients)
        )

    return {
        "precision_5": precision_5,
        "recall_5": recall_5
    }

In [9]:
evaluation_results = {}

for patient_uid, patient_data in task2_results.items():

    relevant_patients = ground_truth[patient_uid]

    # I used bm25 as the other method for task 2
    tfidf_retrieved = patient_data["tfidf"]
    bm25_retrieved = patient_data["bm25"]

    # calculate TF-IDF metrics
    tfidf_metrics = calculate_metrics(
        tfidf_retrieved,
        relevant_patients
    )

    # calculate BM25 metrics
    bm25_metrics = calculate_metrics(
        bm25_retrieved,
        relevant_patients
    )

    # patient evaluation results
    evaluation_results[patient_uid] = {

        "patient_id": patient_data["patient_id"],

        "tfidf": tfidf_metrics,

        "bm25": bm25_metrics
    }


print("Evaluation completed successfully.")
print("Patients evaluated:", len(evaluation_results))

Evaluation completed successfully.
Patients evaluated: 8579


In [10]:
tfidf_precision = []
tfidf_recall = []

bm25_precision = []
bm25_recall = []

excluded_recall_patients = 0

for patient_uid, metrics in evaluation_results.items():

    # TF-IDF
    tfidf_precision.append(
        metrics["tfidf"]["precision_5"]
    )

    if metrics["tfidf"]["recall_5"] is not None:
        tfidf_recall.append(
            metrics["tfidf"]["recall_5"]
        )

    # BM25
    bm25_precision.append(
        metrics["bm25"]["precision_5"]
    )

    if metrics["bm25"]["recall_5"] is not None:
        bm25_recall.append(
            metrics["bm25"]["recall_5"]
        )

    # count patients with no ground-truth similar patients
    if len(ground_truth[patient_uid]) == 0:
        excluded_recall_patients += 1


# averages
average_tfidf_precision = (
    sum(tfidf_precision) / len(tfidf_precision)
)

average_tfidf_recall = (
    sum(tfidf_recall) / len(tfidf_recall)
    if tfidf_recall else None
)

average_bm25_precision = (
    sum(bm25_precision) / len(bm25_precision)
)

average_bm25_recall = (
    sum(bm25_recall) / len(bm25_recall)
    if bm25_recall else None
)

print("Average evaluation metrics calculated.")

Average evaluation metrics calculated.


In [11]:
# save an output file with results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:

    json.dump(
        evaluation_results,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Individual evaluation results saved to:")
print(OUTPUT_FILE)

Individual evaluation results saved to:
/Users/megan/Downloads/PMC_Patients/tfidf_output/task3_evaluation_results.json


In [12]:
evaluation_summary = {

    "total_patients": len(evaluation_results),

    "patients_excluded_from_recall":
        excluded_recall_patients,

    "patients_included_in_recall":
        len(evaluation_results) - excluded_recall_patients,

    "tfidf": {

        "average_precision_5":
            average_tfidf_precision,

        "average_recall_5":
            average_tfidf_recall
    },

    "bm25": {

        "average_precision_5":
            average_bm25_precision,

        "average_recall_5":
            average_bm25_recall
    }
}


with open(SUMMARY_FILE, "w", encoding="utf-8") as file:

    json.dump(
        evaluation_summary,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Evaluation summary saved to:")
print(SUMMARY_FILE)

Evaluation summary saved to:
/Users/megan/Downloads/PMC_Patients/tfidf_output/task3_evaluation_summary.json


In [13]:
print("\n" + "=" * 55)
print("TASK 3: RETRIEVAL EVALUATION REPORT")
print("=" * 55)

print("\nTotal patients evaluated:",
      len(evaluation_results))

print("Patients included in recall:",
      len(tfidf_recall))

print("Patients excluded from recall:",
      excluded_recall_patients)

print("\n" + "-" * 55)

print(f"{'Evaluation Metric':<25}{'TF-IDF':>12}{'BM25':>12}")

print("-" * 55)

print(
    f"{'Average Precision@5':<25}"
    f"{average_tfidf_precision:>12.4f}"
    f"{average_bm25_precision:>12.4f}"
)

if average_tfidf_recall is not None and average_bm25_recall is not None:

    print(
        f"{'Average Recall@5':<25}"
        f"{average_tfidf_recall:>12.4f}"
        f"{average_bm25_recall:>12.4f}"
    )

else:
    print("Average Recall@5 could not be calculated.")

print("-" * 55)

print("\nTask 3 completed successfully!")

print("\nIndividual results:")
print(OUTPUT_FILE)

print("\nSummary results:")
print(SUMMARY_FILE)


TASK 3: RETRIEVAL EVALUATION REPORT

Total patients evaluated: 8579
Patients included in recall: 4802
Patients excluded from recall: 3777

-------------------------------------------------------
Evaluation Metric              TF-IDF        BM25
-------------------------------------------------------
Average Precision@5            0.0078      0.0065
Average Recall@5               0.0332      0.0284
-------------------------------------------------------

Task 3 completed successfully!

Individual results:
/Users/megan/Downloads/PMC_Patients/tfidf_output/task3_evaluation_results.json

Summary results:
/Users/megan/Downloads/PMC_Patients/tfidf_output/task3_evaluation_summary.json
